# LangChain: Agents (Modernized with LangGraph)

## Outline:

* Using built-in LangChain tools: LLM-Math and Wikipedia
* Defining your own tools
* Using `create_react_agent` from `langgraph.prebuilt`

> **Note**: This notebook uses `langgraph.prebuilt.create_react_agent` and modern tool imports instead of the deprecated `initialize_agent`, `AgentType`, `load_tools`, and `create_python_agent`.

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings("ignore")

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [2]:
# Set the model variable
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

## Built-in LangChain tools

In [3]:
from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [4]:
llm = ChatOpenAI(temperature=0, model=llm_model)

In [5]:
# Define tools explicitly (replaces deprecated load_tools)
@tool
def llm_math(question: str) -> str:
    """Useful for answering math questions. Input should be a math expression or question."""
    math_prompt = f"Calculate the following and return only the numeric result: {question}"
    response = llm.invoke(math_prompt)
    return response.content

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools = [llm_math, wikipedia]

In [6]:
# create_react_agent from langgraph (replaces deprecated initialize_agent + AgentType)
agent = create_react_agent(llm, tools)

In [7]:
# .invoke() with messages dict (replaces agent("..."))
result = agent.invoke({"messages": [{"role": "user", "content": "What is the 25% of 300?"}]})
print(result["messages"][-1].content)

25% of 300 is 75.


## Wikipedia example

In [8]:
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].content)

Tom M. Mitchell is the author of the textbook "Machine Learning."


## Python Agent

In [9]:
# Using PythonREPLTool with create_react_agent (replaces deprecated create_python_agent)
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
python_agent = create_react_agent(llm, [python_repl])

In [10]:
customer_list = [["Harrison", "Chase"],
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"],
                 ["Geoff","Fusion"],
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]

In [11]:
result = python_agent.invoke({
    "messages": [{"role": "user", "content": f"Sort these customers by \
last name and then first name \
and print the output: {customer_list}"}]
})
print(result["messages"][-1].content)

Python REPL can execute arbitrary code. Use with caution.


The customers sorted by last name and then first name are:

1. Jen Ayai
2. Lang Chain
3. Harrison Chase
4. Elle Elem
5. Trance Former
6. Geoff Fusion
7. Dolly Too


#### View detailed outputs of the chains

In [12]:
from langchain_core.globals import set_debug
set_debug(True)

result = python_agent.invoke({
    "messages": [{"role": "user", "content": f"Sort these customers by \
last name and then first name \
and print the output: {customer_list}"}]
})
print(result["messages"][-1].content)

set_debug(False)

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "messages": [
    {
      "role": "user",
      "content": "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]"
    }
  ]
}
[chain/start] [chain:LangGraph > chain:agent] Entering Chain run with input:
[inputs]
[chain/start] [chain:LangGraph > chain:agent > chain:call_model] Entering Chain run with input:
[inputs]
[chain/start] [chain:LangGraph > chain:agent > chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:LangGraph > chain:agent > chain:RunnableSequence > chain:Prompt] Entering Chain run with input:
[inputs]
[chain/end] [chain:LangGraph > chain:agent > chain:RunnableSequence > chain:Prompt] s] Exiting Chain run with output:
[outputs]
[llm/start] [chain:LangGraph > chain:agent > chain:RunnableSequence > llm:ChatOpenAI] Ent

## Define your own tool

In [13]:
from datetime import date

In [14]:
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [15]:
# Rebuild agent with custom tool added
agent = create_react_agent(llm, tools + [time])

**Note**: 

The agent will sometimes come to the wrong conclusion (agents are a work in progress!). 

If it does, please try running it again.

In [16]:
try:
    result = agent.invoke({"messages": [{"role": "user", "content": "whats the date today?"}]})
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Exception: {e}")

Today's date is April 9, 2026.
